In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.signal import butter, filtfilt, find_peaks, hilbert
import warnings
import re

In [2]:
# ====== PARAMETRES FILTRE =====

fs = 200  # fréquence d'échantillonnage (Hz)
fc = 5   # fréquence de coupure (Hz)
b, a = butter(4, fc/(fs/2), btype='low')
h = 1/fs  # pas de temps = 0.005 s

# ========================

In [10]:
# ====== PARAMETRES ======
dossier = "Coda_data"

fichiers_S01 = [
    "Gr4_S01_L1_20250011_001.txt",
    "Gr4_S01_L1_20250011_002.txt",
    "Gr4_S01_L1_20250011_003.txt",
    "Gr4_S01_L1_20250011_004.txt",
    "Gr4_S01_L2_20250011_005.txt",
    "Gr4_S01_L2_20250011_006.txt",
    "Gr4_S01_L2_20250011_007.txt",
    "Gr4_S01_L2_20250011_008.txt",
    "Gr4_S01_L3_20250011_009.txt",
    "Gr4_S01_L3_20250011_010.txt",
    "Gr4_S01_L3_20250011_011.txt",
    "Gr4_S01_L3_20250011_012.txt",
    "Gr4_S01_L4_20250011_013.txt",
    "Gr4_S01_L4_20250011_014.txt",
    "Gr4_S01_L4_20250011_015.txt",
    "Gr4_S01_L4_20250011_016.txt",
]

# ====== FILTRE PASSE-BAS ======
fs = 200
fc = 10
b, a = butter(4, fc/(fs/2), btype='low')

# ====== DATAFRAME FINAL ======
resultats = []

# ====== DATAFRAME POUR ANGLE ======
resultats_angle = []

for f in fichiers_S01:
    chemin = os.path.join(dossier, f)

    df = pd.read_csv(chemin, sep="\t", header=2)
    df.columns = df.columns.str.strip()
    df = df.replace("NaN", np.nan)

    colonnes_a_supprimer = [c for c in df.columns if ".V" in c or c.strip() == ""]
    df = df.drop(columns=colonnes_a_supprimer, errors="ignore")

    time_col = "Time" if "Time" in df.columns else "Time (s)"

    pattern = r"Marker 0[5-8]\.[XYZ]"
    colonnes_markers = [c for c in df.columns if re.search(pattern, c)]
    df = df[[time_col] + colonnes_markers]
    df = df.apply(pd.to_numeric, errors="coerce")

    # ====== CENTRE RECTANGLE ======
    cx = ((df["Marker 05.X"] + df["Marker 06.X"])/2 +
          (df["Marker 07.X"] + df["Marker 08.X"])/2)/2

    cy = ((df["Marker 05.Y"] + df["Marker 06.Y"])/2 +
          (df["Marker 07.Y"] + df["Marker 08.Y"])/2)/2

    cz = ((df["Marker 05.Z"] + df["Marker 06.Z"])/2 +
          (df["Marker 07.Z"] + df["Marker 08.Z"])/2)/2

    # ====== MILIEU MARKERS 5-6 ======
    top_x = (df["Marker 05.X"] + df["Marker 06.X"]) / 2
    top_y = (df["Marker 05.Y"] + df["Marker 06.Y"]) / 2

    # ====== INTERPOLATION NaN ======
    for signal in [cx, cy, cz, top_x, top_y]:
        nan_idx = np.isnan(signal)
        if np.any(nan_idx):
            signal[nan_idx] = np.interp(
                np.flatnonzero(nan_idx),
                np.flatnonzero(~nan_idx),
                signal[~nan_idx]
            )

    # ====== FILTRAGE ======
    cx = filtfilt(b, a, cx)
    cy = filtfilt(b, a, cy)
    cz = filtfilt(b, a, cz)

    top_x = filtfilt(b, a, top_x)
    top_y = filtfilt(b, a, top_y)

    # ====== NOM PROPRE ======
    m = re.search(r"Gr4_(S\d+)_(L\d+)_\d+_(\d+)\.txt$", f)
    if m:
        s_part = m.group(1)
        l_part = m.group(2)
        num = str(int(m.group(3)))
        col_base = f"{s_part}_{l_part}_{num}"
    else:
        col_base = f.replace(".txt","")

    # ====== DATAFRAME POSITION (comme avant) ======
    df_res = pd.DataFrame({
        "Time": df[time_col],
        f"{col_base}_X": cx,
        f"{col_base}_Y": cy,
        f"{col_base}_Z": cz
    })

    resultats.append(df_res)

    # ====== DATAFRAME POUR ANGLE ======
    df_angle = pd.DataFrame({
        "Time": df[time_col],
        f"{col_base}_center_X": cx,
        f"{col_base}_center_Y": cy,
        f"{col_base}_top_X": top_x,
        f"{col_base}_top_Y": top_y
    })

    resultats_angle.append(df_angle)


# ====== FUSION DATAFRAME POSITION ======
df_final = resultats[0]
for df in resultats[1:]:
    df_final = df_final.merge(df, on="Time")

# ====== FUSION DATAFRAME ANGLE ======
df_angle_base = resultats_angle[0]
for df in resultats_angle[1:]:
    df_angle_base = df_angle_base.merge(df, on="Time")

# ====== AFFICHAGE ======
print("✅ Fusion terminée")

print("\nColonnes df_final :")
print(df_final.columns.tolist())

print("\nColonnes df_angle_base :")
print(df_angle_base.columns.tolist())

✅ Fusion terminée

Colonnes df_final :
['Time', 'S01_L1_1_X', 'S01_L1_1_Y', 'S01_L1_1_Z', 'S01_L1_2_X', 'S01_L1_2_Y', 'S01_L1_2_Z', 'S01_L1_3_X', 'S01_L1_3_Y', 'S01_L1_3_Z', 'S01_L1_4_X', 'S01_L1_4_Y', 'S01_L1_4_Z', 'S01_L2_5_X', 'S01_L2_5_Y', 'S01_L2_5_Z', 'S01_L2_6_X', 'S01_L2_6_Y', 'S01_L2_6_Z', 'S01_L2_7_X', 'S01_L2_7_Y', 'S01_L2_7_Z', 'S01_L2_8_X', 'S01_L2_8_Y', 'S01_L2_8_Z', 'S01_L3_9_X', 'S01_L3_9_Y', 'S01_L3_9_Z', 'S01_L3_10_X', 'S01_L3_10_Y', 'S01_L3_10_Z', 'S01_L3_11_X', 'S01_L3_11_Y', 'S01_L3_11_Z', 'S01_L3_12_X', 'S01_L3_12_Y', 'S01_L3_12_Z', 'S01_L4_13_X', 'S01_L4_13_Y', 'S01_L4_13_Z', 'S01_L4_14_X', 'S01_L4_14_Y', 'S01_L4_14_Z', 'S01_L4_15_X', 'S01_L4_15_Y', 'S01_L4_15_Z', 'S01_L4_16_X', 'S01_L4_16_Y', 'S01_L4_16_Z']

Colonnes df_angle_base :
['Time', 'S01_L1_1_center_X', 'S01_L1_1_center_Y', 'S01_L1_1_top_X', 'S01_L1_1_top_Y', 'S01_L1_2_center_X', 'S01_L1_2_center_Y', 'S01_L1_2_top_X', 'S01_L1_2_top_Y', 'S01_L1_3_center_X', 'S01_L1_3_center_Y', 'S01_L1_3_top_X', 'S01_L1_

In [30]:
# ====== DOSSIER DES PLOTS ======
dossier_plots = "position_S01_filtre"
os.makedirs(dossier_plots, exist_ok=True)

time_col = "Time"
cols = [c for c in df_final.columns if c != time_col]

# ====== PLOTS ======
for col in cols:

    plt.figure(figsize=(8,3))

    plt.plot(df_final[time_col], df_final[col])

    plt.xlabel("Temps (s)")
    plt.ylabel(col)
    plt.title(f"{col} filtré (<{fc} Hz)")

    ticks = np.arange(
        0,
        float(df_final[time_col].iloc[-1]) + 1,
        5
    )
    plt.xticks(ticks)

    plt.tight_layout()

    # sauvegarde
    nom_fichier = f"{col}.png"
    chemin_plot = os.path.join(dossier_plots, nom_fichier)

    plt.savefig(chemin_plot, dpi=150)
    plt.close()

print(f"✅ Plots sauvegardés dans : {dossier_plots}")

✅ Plots sauvegardés dans : position_S01_filtre


In [4]:
# ====== FONCTION VITESSE CENTREE ======
def vitesse_centre(U, h):
    """
    Dérivée centrée d'ordre 4 :
    u'[i] = (-U[i+2] + 8*U[i+1] - 8*U[i-1] + U[i-2]) / (12*h)
    """
    U = np.array(U, dtype=float)
    N = len(U)
    V = np.zeros_like(U)

    # début
    V[0] = (U[1] - U[0]) / h
    V[1] = (U[2] - U[1]) / h

    # points centraux
    for i in range(2, N-2):
        V[i] = (-U[i+2] + 8*U[i+1] - 8*U[i-1] + U[i-2]) / (12*h)

    # fin
    V[-2] = (U[-2] - U[-3]) / h
    V[-1] = (U[-1] - U[-2]) / h

    return V


# ====== CALCUL DES VITESSES ======

velocities = pd.DataFrame()

time_col = "Time"
velocities[time_col] = df_final[time_col]

cols = [c for c in df_final.columns if c != time_col]

for col in cols:

    velocities[col] = vitesse_centre(
        df_final[col].values,
        h
    )

print("✅ Calcul des vitesses terminé")

print("\nColonnes vitesses :")
print(velocities.columns.tolist())

✅ Calcul des vitesses terminé

Colonnes vitesses :
['Time', 'S01_L1_1_X', 'S01_L1_1_Y', 'S01_L1_1_Z', 'S01_L1_2_X', 'S01_L1_2_Y', 'S01_L1_2_Z', 'S01_L1_3_X', 'S01_L1_3_Y', 'S01_L1_3_Z', 'S01_L1_4_X', 'S01_L1_4_Y', 'S01_L1_4_Z', 'S01_L2_5_X', 'S01_L2_5_Y', 'S01_L2_5_Z', 'S01_L2_6_X', 'S01_L2_6_Y', 'S01_L2_6_Z', 'S01_L2_7_X', 'S01_L2_7_Y', 'S01_L2_7_Z', 'S01_L2_8_X', 'S01_L2_8_Y', 'S01_L2_8_Z', 'S01_L3_9_X', 'S01_L3_9_Y', 'S01_L3_9_Z', 'S01_L3_10_X', 'S01_L3_10_Y', 'S01_L3_10_Z', 'S01_L3_11_X', 'S01_L3_11_Y', 'S01_L3_11_Z', 'S01_L3_12_X', 'S01_L3_12_Y', 'S01_L3_12_Z', 'S01_L4_13_X', 'S01_L4_13_Y', 'S01_L4_13_Z', 'S01_L4_14_X', 'S01_L4_14_Y', 'S01_L4_14_Z', 'S01_L4_15_X', 'S01_L4_15_Y', 'S01_L4_15_Z', 'S01_L4_16_X', 'S01_L4_16_Y', 'S01_L4_16_Z']


In [32]:
# ====== PLOT DES VITESSES (+timing) ======
dossier_plots_vitesse = "vitesse_S01"
os.makedirs(dossier_plots_vitesse, exist_ok=True)

t_min = 7
t_max = 43
mask = (velocities[time_col] >= t_min) & (velocities[time_col] <= t_max)

for col in cols:
    plt.figure(figsize=(8,3))
    plt.plot(velocities[time_col][mask], velocities[col][mask], color='blue')
    plt.xlabel("Temps (s)")
    plt.ylabel(f"Vitesse de {col}")
    plt.title(f"Vitesse centrée filtrée (<{fc} Hz) de {col} ({t_min}–{t_max} s)")
    plt.grid(True)
    plt.tight_layout()

    # sauvegarde
    nom_fichier = f"vitesse_{col.replace(' ', '_')}.png"
    chemin_plot = os.path.join(dossier_plots_vitesse, nom_fichier)
    plt.savefig(chemin_plot, dpi=150)
    plt.close()

print(f"✅ Plot terminé et sauvegardé dans \"{dossier_plots_vitesse}\"")

✅ Plot terminé et sauvegardé dans "vitesse_S01"


In [5]:
# ====== PARAMETRES ======
t_min = 10
t_max = 43

distance_pts = 200
prominence_pts = 5

time_col = "Time"
cols = [c for c in df_final.columns if c != time_col]

# ====== SELECTION INTERVALLE TEMPS ======
mask = (df_final[time_col] >= t_min) & (df_final[time_col] <= t_max)
time_segment = df_final[time_col][mask]

# ====== STOCKAGE ======
all_min_times = {}
all_max_times = {}

for col in cols:

    signal = df_final[col][mask].values
    time_vals = time_segment.values

    # ====== DETECTION MINIMA ======
    peaks_min, _ = find_peaks(
        -signal,
        distance=distance_pts,
        prominence=prominence_pts
    )

    minima_times = list(time_vals[peaks_min])

    # enlever premier et dernier
    if len(minima_times) > 2:
        minima_times = minima_times[1:-1]

    all_min_times[col] = minima_times


    # ====== DETECTION MAXIMA ======
    peaks_max, _ = find_peaks(
        signal,
        distance=distance_pts,
        prominence=prominence_pts
    )

    maxima_times = list(time_vals[peaks_max])

    # enlever premier et dernier
    if len(maxima_times) > 2:
        maxima_times = maxima_times[1:-1]

    all_max_times[col] = maxima_times


    # ====== PRINT INFO ======
    print(f"\n📊 {col}")
    print(f"Minima détectés (après suppression bords) : {len(minima_times)}")
    print(f"Maxima détectés (après suppression bords) : {len(maxima_times)}")


# ====== DATAFRAME MINIMA ======
max_len_min = max(len(v) for v in all_min_times.values())

minima_times_df = pd.DataFrame({
    col: pd.Series(v + [np.nan]*(max_len_min - len(v)))
    for col, v in all_min_times.items()
})


# ====== DATAFRAME MAXIMA ======
max_len_max = max(len(v) for v in all_max_times.values())

maxima_times_df = pd.DataFrame({
    col: pd.Series(v + [np.nan]*(max_len_max - len(v)))
    for col, v in all_max_times.items()
})


print(f"\n✅ DataFrame des minima : {minima_times_df.shape[0]} x {minima_times_df.shape[1]}")
print(f"✅ DataFrame des maxima : {maxima_times_df.shape[0]} x {maxima_times_df.shape[1]}")


📊 S01_L1_1_X
Minima détectés (après suppression bords) : 15
Maxima détectés (après suppression bords) : 15

📊 S01_L1_1_Y
Minima détectés (après suppression bords) : 13
Maxima détectés (après suppression bords) : 14

📊 S01_L1_1_Z
Minima détectés (après suppression bords) : 14
Maxima détectés (après suppression bords) : 13

📊 S01_L1_2_X
Minima détectés (après suppression bords) : 11
Maxima détectés (après suppression bords) : 12

📊 S01_L1_2_Y
Minima détectés (après suppression bords) : 14
Maxima détectés (après suppression bords) : 15

📊 S01_L1_2_Z
Minima détectés (après suppression bords) : 14
Maxima détectés (après suppression bords) : 12

📊 S01_L1_3_X
Minima détectés (après suppression bords) : 14
Maxima détectés (après suppression bords) : 13

📊 S01_L1_3_Y
Minima détectés (après suppression bords) : 13
Maxima détectés (après suppression bords) : 14

📊 S01_L1_3_Z
Minima détectés (après suppression bords) : 14
Maxima détectés (après suppression bords) : 13

📊 S01_L1_4_X
Minima détecté

In [7]:
# ====== PARAMETRES ======
dossier_plots_vitesse = "vitesse_S01_extrema"
os.makedirs(dossier_plots_vitesse, exist_ok=True)

time_col = "Time"
cols = [c for c in velocities.columns if c != time_col]

t_min = 7
t_max = 43

print(f"Velocities DataFrame : {velocities.shape[0]} lignes x {velocities.shape[1]} colonnes")
print(f"Minima Times DataFrame : {minima_times_df.shape[0]} lignes x {minima_times_df.shape[1]} colonnes")
print(f"Maxima Times DataFrame : {maxima_times_df.shape[0]} lignes x {maxima_times_df.shape[1]} colonnes\n")


for col in cols:

    # ====== SELECTION DE L'INTERVALLE ======
    mask = (velocities[time_col] >= t_min) & (velocities[time_col] <= t_max)

    time_vals = velocities[time_col][mask].values
    signal = velocities[col][mask].values

    plt.figure(figsize=(8,3))
    plt.plot(time_vals, signal, color='blue', label="Vitesse")


    # ====== MINIMA ======
    if col in minima_times_df.columns:

        peak_times = minima_times_df[col].dropna().values
        peak_times = peak_times[(peak_times >= t_min) & (peak_times <= t_max)]

        if len(peak_times) > 0:

            peak_indices = []
            for t in peak_times:
                idx = np.argmin(np.abs(time_vals - t))
                peak_indices.append(idx)

            peak_values = signal[peak_indices]

            plt.scatter(time_vals[peak_indices], peak_values,
                        color='red', s=30, label="Minima")


    # ====== MAXIMA ======
    if col in maxima_times_df.columns:

        peak_times = maxima_times_df[col].dropna().values
        peak_times = peak_times[(peak_times >= t_min) & (peak_times <= t_max)]

        if len(peak_times) > 0:

            peak_indices = []
            for t in peak_times:
                idx = np.argmin(np.abs(time_vals - t))
                peak_indices.append(idx)

            peak_values = signal[peak_indices]

            plt.scatter(time_vals[peak_indices], peak_values,
                        color='green', s=30, label="Maxima")


    # ====== FIGURE ======
    plt.xlabel("Temps (s)")
    plt.ylabel(f"Vitesse {col}")
    plt.title(f"Vitesse filtrée (<{fc} Hz) de {col} ({t_min}-{t_max}s)")
    plt.grid(True)
    plt.legend()
    plt.xlim(t_min, t_max)
    plt.tight_layout()


    # ====== SAUVEGARDE ======
    nom_fichier = f"vitesse_{col}.png"
    chemin_plot = os.path.join(dossier_plots_vitesse, nom_fichier)

    plt.savefig(chemin_plot, dpi=150)
    plt.close()


print(f"\n✅ Plots vitesse sauvegardés dans : {dossier_plots_vitesse}")

Velocities DataFrame : 8601 lignes x 49 colonnes
Minima Times DataFrame : 16 lignes x 48 colonnes
Maxima Times DataFrame : 15 lignes x 48 colonnes


✅ Plots vitesse sauvegardés dans : vitesse_S01_extrema


In [8]:
# ====== PARAMETRES ======
time_col = "Time"
cols = [c for c in df_final.columns if c != time_col]

dossier_plots_superp = "superp_S01"
os.makedirs(dossier_plots_superp, exist_ok=True)

cmap = plt.cm.viridis

# ====== SUPERPOSITION DES CYCLES ======
for col in cols:

    if col not in minima_times_df.columns:
        continue

    minima_times = minima_times_df[col].dropna().values

    if len(minima_times) < 2:
        continue

    fig, ax = plt.subplots(figsize=(8,4))

    n_cycles = len(minima_times) - 1

    for i in range(n_cycles):

        t_start = minima_times[i]
        t_end = minima_times[i+1]

        mask = (df_final[time_col] >= t_start) & (df_final[time_col] <= t_end)

        segment_time = df_final[time_col][mask].values - t_start
        segment_signal = df_final[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / n_cycles)

        ax.plot(segment_time, segment_signal, color=color)

    ax.set_xlabel("Temps depuis minima (s)")
    ax.set_ylabel(col)
    ax.set_title(f"Superposition des cycles : {col}")
    ax.grid(True)

    # ====== COLORBAR ======
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=n_cycles))
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("Numéro du cycle\n(premiers → derniers)")

    fig.tight_layout()

    # ====== SAUVEGARDE ======
    nom_fichier = f"superp_{col}.png"
    chemin_plot = os.path.join(dossier_plots_superp, nom_fichier)

    fig.savefig(chemin_plot, dpi=150)
    plt.close(fig)

print(f"\n✅ Graphiques sauvegardés dans : {dossier_plots_superp}")


✅ Graphiques sauvegardés dans : superp_S01


In [9]:
# ====== PARAMETRES ======
time_col = "Time"
cols = [c for c in velocities.columns if c != time_col]

dossier_plots_superv = "superv_S01"
os.makedirs(dossier_plots_superv, exist_ok=True)

cmap = plt.cm.viridis

# ====== SUPERPOSITION DES VITESSES PAR CYCLE ======
for col in cols:

    if col not in minima_times_df.columns:
        continue

    minima_times = minima_times_df[col].dropna().values

    if len(minima_times) < 2:
        continue

    fig, ax = plt.subplots(figsize=(8,4))

    n_cycles = len(minima_times) - 1

    for i in range(n_cycles):

        t_start = minima_times[i]
        t_end = minima_times[i+1]

        # intervalle du cycle
        mask = (velocities[time_col] >= t_start) & (velocities[time_col] <= t_end)

        segment_time = velocities[time_col][mask].values - t_start
        segment_signal = velocities[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / n_cycles)

        ax.plot(segment_time, segment_signal, color=color)

    ax.set_xlabel("Temps depuis minima (s)")
    ax.set_ylabel(f"Vitesse {col}")
    ax.set_title(f"Superposition des vitesses : {col}")

    ax.grid(True)

    # ====== COLORBAR ======
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=n_cycles))
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("Numéro du cycle\n(premiers → derniers)")

    fig.tight_layout()

    # ====== SAUVEGARDE ======
    nom_fichier = f"superv_{col}.png"
    chemin_plot = os.path.join(dossier_plots_superv, nom_fichier)

    fig.savefig(chemin_plot, dpi=150)
    plt.close(fig)

print(f"\n✅ Graphiques de vitesses sauvegardés dans : {dossier_plots_superv}")


✅ Graphiques de vitesses sauvegardés dans : superv_S01


In [10]:
# ====== PARAMETRES ======
time_col = "Time"
cols = [c for c in df_final.columns if c != time_col]

dossier_plots_sep = "superp_sep_S01"
os.makedirs(dossier_plots_sep, exist_ok=True)

cmap = plt.cm.viridis

# ====== SUPERPOSITION ASCENDANT / DESCENDANT ======
for col in cols:

    if col not in minima_times_df.columns or col not in maxima_times_df.columns:
        continue

    minima_times = minima_times_df[col].dropna().values
    maxima_times = maxima_times_df[col].dropna().values

    if len(minima_times) < 1 or len(maxima_times) < 1:
        continue

    # ====== FIGURES ======
    fig_up, ax_up = plt.subplots(figsize=(8,4))
    fig_down, ax_down = plt.subplots(figsize=(8,4))

    cycles_up = []
    cycles_down = []

    # ====== ASSOCIATION MIN/MAX ======
    for t_min in minima_times:

        max_after = maxima_times[maxima_times > t_min]
        if len(max_after) == 0:
            continue

        t_max = max_after[0]

        cycles_up.append((t_min, t_max))

    for t_max in maxima_times:

        min_after = minima_times[minima_times > t_max]
        if len(min_after) == 0:
            continue

        t_min = min_after[0]

        cycles_down.append((t_max, t_min))


    # ====== PHASE ASCENDANTE ======
    n_up = len(cycles_up)

    for i, (t_start, t_end) in enumerate(cycles_up):

        mask = (df_final[time_col] >= t_start) & (df_final[time_col] <= t_end)

        segment_time = df_final[time_col][mask].values - t_start
        segment_signal = df_final[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / n_up)

        ax_up.plot(segment_time, segment_signal, color=color)


    # ====== PHASE DESCENDANTE ======
    n_down = len(cycles_down)

    for i, (t_start, t_end) in enumerate(cycles_down):

        mask = (df_final[time_col] >= t_start) & (df_final[time_col] <= t_end)

        segment_time = df_final[time_col][mask].values - t_start
        segment_signal = df_final[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / n_down)

        ax_down.plot(segment_time, segment_signal, color=color)


    # ====== STYLE ======
    ax_up.set_xlabel("Temps depuis minima (s)")
    ax_up.set_ylabel(col)
    ax_up.set_title(f"Phases ascendantes : {col}")
    ax_up.grid(True)

    ax_down.set_xlabel("Temps depuis maxima (s)")
    ax_down.set_ylabel(col)
    ax_down.set_title(f"Phases descendantes : {col}")
    ax_down.grid(True)


    # ====== COLORBARS ======
    sm_up = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=max(1,n_up)))
    sm_up.set_array([])
    fig_up.colorbar(sm_up, ax=ax_up).set_label("Numéro cycle")

    sm_down = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=max(1,n_down)))
    sm_down.set_array([])
    fig_down.colorbar(sm_down, ax=ax_down).set_label("Numéro cycle")

    fig_up.tight_layout()
    fig_down.tight_layout()

    # ====== SAUVEGARDE ======
    fig_up.savefig(os.path.join(dossier_plots_sep, f"asc_{col}.png"), dpi=150)
    fig_down.savefig(os.path.join(dossier_plots_sep, f"desc_{col}.png"), dpi=150)

    plt.close(fig_up)
    plt.close(fig_down)

print(f"\n✅ Graphiques sauvegardés dans : {dossier_plots_sep}")


✅ Graphiques sauvegardés dans : superp_sep_S01


In [11]:
# ====== PARAMETRES ======
time_col = "Time"
cols = [c for c in velocities.columns if c != time_col]

dossier_plots_sep = "superv_sep_S01"
os.makedirs(dossier_plots_sep, exist_ok=True)

cmap = plt.cm.viridis

# ====== SUPERPOSITION ASCENDANT / DESCENDANT ======
for col in cols:

    if col not in minima_times_df.columns or col not in maxima_times_df.columns:
        continue

    minima_times = minima_times_df[col].dropna().values
    maxima_times = maxima_times_df[col].dropna().values

    if len(minima_times) < 1 or len(maxima_times) < 1:
        continue

    fig_up, ax_up = plt.subplots(figsize=(8,4))
    fig_down, ax_down = plt.subplots(figsize=(8,4))

    cycles_up = []
    cycles_down = []

    # ====== ASSOCIER MIN -> MAX ======
    for t_min in minima_times:

        max_after = maxima_times[maxima_times > t_min]
        if len(max_after) == 0:
            continue

        t_max = max_after[0]
        cycles_up.append((t_min, t_max))

    # ====== ASSOCIER MAX -> MIN ======
    for t_max in maxima_times:

        min_after = minima_times[minima_times > t_max]
        if len(min_after) == 0:
            continue

        t_min = min_after[0]
        cycles_down.append((t_max, t_min))


    # ====== PHASE ASCENDANTE ======
    n_up = len(cycles_up)

    for i, (t_start, t_end) in enumerate(cycles_up):

        mask = (velocities[time_col] >= t_start) & (velocities[time_col] <= t_end)

        segment_time = velocities[time_col][mask].values - t_start
        segment_signal = velocities[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / max(1, n_up))
        ax_up.plot(segment_time, segment_signal, color=color)


    # ====== PHASE DESCENDANTE ======
    n_down = len(cycles_down)

    for i, (t_start, t_end) in enumerate(cycles_down):

        mask = (velocities[time_col] >= t_start) & (velocities[time_col] <= t_end)

        segment_time = velocities[time_col][mask].values - t_start
        segment_signal = velocities[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / max(1, n_down))
        ax_down.plot(segment_time, segment_signal, color=color)


    # ====== STYLE ======
    ax_up.set_xlabel("Temps depuis minima (s)")
    ax_up.set_ylabel(f"Vitesse {col}")
    ax_up.set_title(f"Phases ascendantes : {col}")
    ax_up.grid(True)

    ax_down.set_xlabel("Temps depuis maxima (s)")
    ax_down.set_ylabel(f"Vitesse {col}")
    ax_down.set_title(f"Phases descendantes : {col}")
    ax_down.grid(True)


    # ====== COLORBARS ======
    sm_up = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=max(1,n_up)))
    sm_up.set_array([])
    fig_up.colorbar(sm_up, ax=ax_up).set_label("Numéro cycle")

    sm_down = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=max(1,n_down)))
    sm_down.set_array([])
    fig_down.colorbar(sm_down, ax=ax_down).set_label("Numéro cycle")

    fig_up.tight_layout()
    fig_down.tight_layout()


    # ====== SAUVEGARDE ======
    fig_up.savefig(os.path.join(dossier_plots_sep, f"asc_v_{col}.png"), dpi=150)
    fig_down.savefig(os.path.join(dossier_plots_sep, f"desc_v_{col}.png"), dpi=150)

    plt.close(fig_up)
    plt.close(fig_down)


print(f"\n✅ Graphiques de vitesses sauvegardés dans : {dossier_plots_sep}")


✅ Graphiques de vitesses sauvegardés dans : superv_sep_S01


In [23]:
# ====== PARAMETRES TEMPORELS ======
t_min = 10
t_max = 40

# ====== DOSSIER DE SORTIE ======
dossier_save = "angle_manipu_S01"
os.makedirs(dossier_save, exist_ok=True)

time_col = "Time"

# ====== CONDITIONS DISPONIBLES ======
bases = sorted(set([c.split("_center")[0] for c in df_angle_base.columns if "_center_X" in c]))

# ====== BOUCLE SUR CHAQUE FICHIER ======
for base in bases:

    center_x = df_angle_base[f"{base}_center_X"].values
    center_y = df_angle_base[f"{base}_center_Y"].values

    top_x = df_angle_base[f"{base}_top_X"].values
    top_y = df_angle_base[f"{base}_top_Y"].values

    # ===== vecteur centre → haut =====
    vx = top_x - center_x
    vy = top_y - center_y

    # ===== angle (positif vers gauche) =====
    angle = -np.degrees(np.arctan2(vx, vy))

    angle_df = pd.DataFrame({
        "Time": df_angle_base["Time"],
        "Angle": angle
    })

    # ===== SELECTION INTERVALLE TEMPS =====
    mask = (angle_df["Time"] >= t_min) & (angle_df["Time"] <= t_max)

    t = angle_df["Time"][mask].values
    a = angle_df["Angle"][mask].values

    if len(t) == 0:
        continue

    # ===== FIGURE =====
    fig, ax = plt.subplots(figsize=(8,4))

    ax.plot(t, a)

    # ===== STYLE =====
    ax.set_xlabel("Temps (s)")
    ax.set_ylabel("Angle du rectangle (°)")
    ax.set_title(f"Angle rectangle : {base}")
    ax.grid(True)

    fig.tight_layout()

    # ===== SAUVEGARDE =====
    save_path = os.path.join(dossier_save, f"angle_time_{base}.png")
    fig.savefig(save_path, dpi=150)

    plt.close(fig)

print(f"\n✅ Graphiques sauvegardés dans : {dossier_save}")


✅ Graphiques sauvegardés dans : angle_manipu_S01


In [15]:
# ====== DOSSIER DE SORTIE ======
dossier_save = "supera_manipu_S01"
os.makedirs(dossier_save, exist_ok=True)

time_col = "Time"
cmap = plt.cm.viridis

# ====== CONDITIONS DISPONIBLES ======
bases = sorted(set([c.split("_center")[0] for c in df_angle_base.columns if "_center_X" in c]))

# ====== BOUCLE SUR CHAQUE FICHIER ======
for base in bases:

    if base+"_X" not in minima_times_df.columns:
        continue

    center_x = df_angle_base[f"{base}_center_X"].values
    center_y = df_angle_base[f"{base}_center_Y"].values

    top_x = df_angle_base[f"{base}_top_X"].values
    top_y = df_angle_base[f"{base}_top_Y"].values

    # ===== vecteur centre → haut =====
    vx = top_x - center_x
    vy = top_y - center_y

    # ===== angle (positif vers gauche) =====
    angle = -np.degrees(np.arctan2(vx, vy))

    angle_df = pd.DataFrame({
        "Time": df_angle_base["Time"],
        "Angle": angle
    })

    minima_times = minima_times_df[base+"_X"].dropna().values

    if len(minima_times) < 2:
        continue

    # ===== FIGURE =====
    fig, ax = plt.subplots(figsize=(8,4))

    n_cycles = len(minima_times) - 1

    for i in range(n_cycles):

        t_start = minima_times[i]
        t_end = minima_times[i+1]

        mask = (angle_df["Time"] >= t_start) & (angle_df["Time"] <= t_end)

        t = angle_df["Time"][mask].values
        a = angle_df["Angle"][mask].values

        if len(t) == 0:
            continue

        t_norm = (t - t_start) / (t_end - t_start)

        color = cmap(i / max(1,n_cycles))

        ax.plot(t_norm, a, color=color)

    # ===== STYLE =====
    ax.set_xlabel("Progression du cycle")
    ax.set_ylabel("Angle du rectangle (°)")
    ax.set_title(f"Superposition cycles angle : {base}")
    ax.grid(True)

    # ===== COLORBAR =====
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=max(1,n_cycles)))
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("Numéro cycle")

    fig.tight_layout()

    # ===== SAUVEGARDE =====
    save_path = os.path.join(dossier_save, f"angle_cycles_{base}.png")
    fig.savefig(save_path, dpi=150)

    plt.close(fig)

print(f"\n✅ Graphiques sauvegardés dans : {dossier_save}")


✅ Graphiques sauvegardés dans : supera_manipu_S01
